## Additional Exploration

This Jupyter Notebook takes a closer look at the indicators Landcover/Landuse that come with LUCAS data, the Corine Landcover data for our set of points as well as Soil Type information that was extracted from BGR (Soil Regions of the European Union and Adjacent Countries 1 : 5 000 000). The script shows frequency tables for these indicators. Pivot tables can also be created, see Step 3.

After exploring the pivot tables it is clear, that the amount of possible categories/values (i.e. forest, agriculture, water or fallow) per indicator (i.e. landcover) needs to be reduced in the dataset, so that ML later has only few categories to deal with (3 to 5). That means some rows and also columns will we be deleted, Step 4. is a suggestion and shows a possible outcome of such reductions. Finally a first visualisation is done.

OUTPUT: df_SHI_with_LC_LU_reduced.csv

### Content
1. Merging Points & SHI with independent indicators (landcover, landuse, ...)
2. Counting occurences per indicator (frequency tables)
3. Pivot or Cross Tables
4. Drop Columns and Categories (suggestion)
5. Visualisation: Box-Whisker-Plots for SHI/Landcover/Landuse

In [ ]:
import pandas as pd
import csv
import openpyxl
import seaborn as sns
import matplotlib.pyplot as plt
import textwrap
#import numpy as np

### 1. Merging Points & SHI with independent indicators (landcover, landuse, ...)

In [ ]:
# load data
df_shi_lc_lu = pd.read_csv("03_output/df_with_SHI.csv")
df_indep_indicators = pd.read_csv("01_input/Indep_indicators.csv")

# drop some columns not needed now
df_shi_lc_lu.drop(columns=['Clay_SOC_score', 'OC_score', 'P_score', 'N_score',
       'K_score', 'bulk_density_score', 'EC_score', 'erosion_score',
       'pH_score'], inplace=True)

# merge
df_merged = df_shi_lc_lu.merge(
    df_indep_indicators,
    on='POINT_ID',
    how='left',
    suffixes=('', '_ii')  # avoids clashes if columns match
)

# replace all missing values with NaN
df_merged = (
    df_merged
    .replace(r'^\s*$', pd.NA, regex=True)
    .fillna('NO DATA')
)

# reorder columns
df_merged = df_merged.loc[:, ['POINT_ID', 'SHI', 'hoehe_m', 'Landuse', 'Landcover', 'Corine_Landcover', 'DO_RSG1', 'DO_MAT1']]

# save & check
# df_merged.to_csv("03_output/df_points_indep_indicators.csv", index=False)
df_merged.head(15)

### 2. Counting occurences per indicator

In [ ]:
indicators_columns = ['Landuse', 'Landcover', 'Corine_Landcover', 'DO_RSG1', 'DO_MAT1']

for column in indicators_columns:
    counts = df_merged[column].value_counts()

    print(f"\n=== {column} ({counts.size} categories) ===")

    for category, count in counts.items():
        print(f"  {category:<58} {count}")

### 3. Pivot or Cross Tables

Crosstables can be created and printed or exportet as csv or xlsx here. From experience it is a better approach to import a csv directly into Excel and create a pivot table there. Functionality is better, columns can be hidden or shown and counted values will adapt. To do so just import df_points_indep_indicators.csv in Excel and use pivot table function there.

In [ ]:
# Example of creating crosstable, use the two columns you are interested in cross counting
# some possible columns: 'Landuse', 'Landcover', 'Corine_Landcover', 'DO_RSG1', 'DO_MAT1'
df = df_merged.copy()

crosstab = pd.crosstab(df['Landcover'], df['Corine_Landcover'], margins=True)
print(crosstab)

# this file can be used to create pivot tables in Excel
# df.to_excel("03_output/df_points_indep_indicators.xlsx", index=False)

### 4. Drop Columns and Categories (suggestion)

After inspecting the pivot tables:
- keep Landcover and Landuse but reduce to certain accepted categories
- drop Corine_Landcover, no advantages to Landcover (LUCAS)
- drop the soil type and material columns, too much variety, too many categories with regard to ML


In [ ]:

LC_to_keep = ['Bareland', 'Cropland', 'Grassland', 'Shrubland', 'Woodland']
LU_to_keep = ['Agriculture (excluding fallow land and kitchen gardens)', 'Forestry', 'Semi-natural and natural areas not in use', 'Fallow land']
columns_to_drop =  ['Corine_Landcover', 'DO_RSG1', 'DO_MAT1']

# filter the dataframe, create a new one
df_LC_LU_reduced = df[
    df['Landcover'].isin(LC_to_keep)
    & df['Landuse'].isin(LU_to_keep)
]

# check
indicators_columns = ['Landuse', 'Landcover']
for column in indicators_columns:
    counts = df_LC_LU_reduced[column].value_counts()

    print(f"\n=== {column} === Total: {counts.sum()} ===")
    print(counts)
    print()

# drop the columns not needed
df_LC_LU_reduced.drop(columns=columns_to_drop, inplace=True)

# check again
print(f"Size dataset before: {len(df)}")
print(f"Size dataset after reduction: {len(df_LC_LU_reduced)}")

df_LC_LU_reduced.to_csv("03_output/df_SHI_with_LC_LU_reduced.csv", index=False)
df_LC_LU_reduced.head()


### 5. Visualisation: Box-Whisker-Plots for SHI/Landcover/Landuse

Boxplots:
- Median (middle line)
- Interquartile range (IQR) => the box: it is the middle 50% of the data
- Spread / Variability (whiskers)
- outliers (points outside whiskers)

In [ ]:
# Figure 1

plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df_LC_LU_reduced,
    x='Landcover',
    y='SHI',
    hue='Landcover', # required now for palette
    palette='viridis',
    legend=False
)

plt.title("SHI by Landcover")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()



# Figure 2

order = df_LC_LU_reduced['Landuse'].unique() # needed for long label wrapping

plt.figure(figsize=(12, 6))

ax = sns.boxplot(
    data=df_LC_LU_reduced,
    x='Landuse',
    y='SHI',
    palette='BrBG',
    hue='Landuse', # required now for palette
)

# wrap existing tick labels
plt.setp(
    ax.get_xticklabels(),
    rotation=0,
    ha='center'
)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([textwrap.fill(x, 20) for x in order])

plt.title("SHI by Landuse")
plt.tight_layout()
plt.show()


